In [1]:
import pandas as pd

MINIO_ENDPOINT = "http://169.58.96.176:9000"
MINIO_KEY = "admin"
MINIO_SECRET = "adminGrupo3"
BUCKET_NAME = "programacion-ciencia-datos-g3"

storage_opts = {
    "key": MINIO_KEY,
    "secret": MINIO_SECRET,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT}
}

print("📥 Exploración inicial utilizando Chunking (Capa Bronze)...")

CHUNK_SIZE = 50000
total_filas_trans = 0
total_filas_id = 0
columnas_trans = None

# 1. Exploración por bloques de train_transaction.csv
for i, chunk in enumerate(pd.read_csv(
    f"s3://{BUCKET_NAME}/bronze/train_transaction.csv", 
    chunksize=CHUNK_SIZE, 
    storage_options=storage_opts
)):
    if i == 0:
        print(f"\n--- BLOQUE {i+1} (Muestra inicial para head y tipos) ---")
        display(chunk[['TransactionID', 'isFraud', 'TransactionAmt', 'card4']].head())
        columnas_trans = chunk.columns.tolist()
        
    total_filas_trans += len(chunk)
    # Opcional: detenernos tras unos bloques si solo queremos explorar rápido
    if i >= 5: 
        print(f"... (exploración parcial por chunks en curso, procesados {i+1} bloques)")

# 2. Exploración por bloques de train_identity.csv
for i, chunk_id in enumerate(pd.read_csv(
    f"s3://{BUCKET_NAME}/bronze/train_identity.csv", 
    chunksize=CHUNK_SIZE, 
    storage_options=storage_opts
)):
    total_filas_id += len(chunk_id)

print("\n=== REVISIÓN DE DIMENSIONES TOTALES (Vía Chunking) ===")
print(f"   • Total de registros en train_transaction: {total_filas_trans:,} filas")
print(f"   • Total de columnas en train_transaction:  {len(columnas_trans)} columnas")
print(f"   • Total de registros en train_identity:    {total_filas_id:,} filas")



📥 Exploración inicial utilizando Chunking (Capa Bronze)...

--- BLOQUE 1 (Muestra inicial para head y tipos) ---


,TransactionID,isFraud,TransactionAmt,card4
0,2987000,0,68.5,discover
1,2987001,0,29.0,mastercard
2,2987002,0,59.0,visa
3,2987003,0,50.0,mastercard
4,2987004,0,50.0,mastercard


... (exploración parcial por chunks en curso, procesados 6 bloques)
... (exploración parcial por chunks en curso, procesados 7 bloques)
... (exploración parcial por chunks en curso, procesados 8 bloques)
... (exploración parcial por chunks en curso, procesados 9 bloques)
... (exploración parcial por chunks en curso, procesados 10 bloques)
... (exploración parcial por chunks en curso, procesados 11 bloques)
... (exploración parcial por chunks en curso, procesados 12 bloques)

=== REVISIÓN DE DIMENSIONES TOTALES (Vía Chunking) ===
   • Total de registros en train_transaction: 590,540 filas
   • Total de columnas en train_transaction:  394 columnas
   • Total de registros en train_identity:    144,233 filas


In [2]:
import pandas as pd

# Configuración de conexión a tu MinIO en el VPS
MINIO_ENDPOINT = "http://169.58.96.176:9000"
MINIO_KEY = "admin"      # Reemplaza con tu Access Key real
MINIO_SECRET = "adminGrupo3"  # Reemplaza con tu Secret Key real
BUCKET_NAME = "programacion-ciencia-datos-g3"

storage_opts = {
    "key": MINIO_KEY,
    "secret": MINIO_SECRET,
    "client_kwargs": {"endpoint_url": MINIO_ENDPOINT}
}

print("📥 Cargando datos desde la Capa Bronze (MinIO) para exploración...")

# Carga de una muestra optimizada para la exploración inicial
df_trans = pd.read_csv(
    f"s3://{BUCKET_NAME}/bronze/train_transaction.csv", 
    
    storage_options=storage_opts
)
df_id = pd.read_csv(
    f"s3://{BUCKET_NAME}/bronze/train_identity.csv", 
    
    storage_options=storage_opts
)

print("¡Datos cargados con éxito!\n")

# ==============================================================================
# 5.1 EXPLORACIÓN INICIAL DE DATOS (MÉTODOS EXIGIDOS)
# ==============================================================================

print("="*60)
print("1️⃣ REVISIÓN DE DIMENSIONES")
print("="*60)
print(f"Dimensiones de train_transaction: {df_trans.shape[0]:,} filas y {df_trans.shape[1]} columnas")
print(f"Dimensiones de train_identity:    {df_id.shape[0]:,} filas y {df_id.shape[1]} columnas\n")

print("="*60)
print("2️⃣ APLICACIÓN DE head()")
print("="*60)
print("Mostrando los primeros 5 registros de transacciones clave:")
display(df_trans[['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt', 'ProductCD', 'card4', 'card6']].head())

print("\n" + "="*60)
print("3️⃣ APLICACIÓN DE info()")
print("="*60)
print("Estructura general y consumo de memoria del dataset principal:")
df_trans.info(verbose=True, memory_usage="deep")

print("\n" + "="*60)
print("4️⃣ IDENTIFICACIÓN DE TIPOS DE DATOS")
print("="*60)
conteo_tipos = df_trans.dtypes.value_counts()
print("Distribución de tipos de datos en las columnas:")
print(conteo_tipos)

print("\n" + "="*60)
print("5️⃣ APLICACIÓN DE describe()")
print("="*60)
print("Estadísticas descriptivas de variables cuantitativas principales:")
columnas_numericas = ['TransactionAmt', 'isFraud', 'card1', 'dist1','TransactionDT']
display(df_trans[columnas_numericas].describe().T)

📥 Cargando datos desde la Capa Bronze (MinIO) para exploración...
¡Datos cargados con éxito!

1️⃣ REVISIÓN DE DIMENSIONES
Dimensiones de train_transaction: 590,540 filas y 394 columnas
Dimensiones de train_identity:    144,233 filas y 41 columnas

2️⃣ APLICACIÓN DE head()
Mostrando los primeros 5 registros de transacciones clave:


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card4,card6
0,2987000,0,86400,68.5,W,discover,credit
1,2987001,0,86401,29.0,W,mastercard,credit
2,2987002,0,86469,59.0,W,visa,debit
3,2987003,0,86499,50.0,W,mastercard,debit
4,2987004,0,86506,50.0,H,mastercard,credit



3️⃣ APLICACIÓN DE info()
Estructura general y consumo de memoria del dataset principal:
<class 'pandas.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Data columns (total 394 columns):
 #    Column          Dtype  
---   ------          -----  
 0    TransactionID   int64  
 1    isFraud         int64  
 2    TransactionDT   int64  
 3    TransactionAmt  float64
 4    ProductCD       str    
 5    card1           int64  
 6    card2           float64
 7    card3           float64
 8    card4           str    
 9    card5           float64
 10   card6           str    
 11   addr1           float64
 12   addr2           float64
 13   dist1           float64
 14   dist2           float64
 15   P_emaildomain   str    
 16   R_emaildomain   str    
 17   C1              float64
 18   C2              float64
 19   C3              float64
 20   C4              float64
 21   C5              float64
 22   C6              float64
 23   C7              float64
 24   C8              float64


,count,mean,std,min,25%,50%,75%,max
TransactionAmt,590540.0,1.350272e+02,2.391625e+02,0.251,43.321,68.769,125.0,3.193739e+04
isFraud,590540.0,3.499001e-02,1.837546e-01,0.000,0.000,0.000,0.0,1.000000e+00
card1,590540.0,9.898735e+03,4.901170e+03,1000.000,6019.000,9678.000,14184.0,1.839600e+04
dist1,238269.0,1.185022e+02,3.718720e+02,0.000,3.000,8.000,24.0,1.028600e+04
TransactionDT,590540.0,7.372311e+06,4.617224e+06,86400.000,3027057.750,7306527.500,11246620.0,1.581113e+07
